In [ ]:
import sys

sys.path.append("..")

from src.data import nysm_data
from src.data import hrrr_data
from src.processing import get_closest_nysm_stations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import statistics as st
from datetime import datetime
import cartopy.crs as crs
import geopandas as gpd
import matplotlib.patches as mpatches
import cartopy.feature as cfeature
import matplotlib.image as mpimg

In [ ]:
# load nysm
nysm_df = nysm_data.load_nysm_data(gfs=False)
# load hrrr
hrrr_df = hrrr_data.read_hrrr_data(str(1).zfill(2))

In [ ]:
# station
station = "VOOR"
stations = get_closest_nysm_stations.get_closest_stations(nysm_df, 4, station, "HRRR")

In [ ]:
filtered_nysm = nysm_df[nysm_df["station"].isin(stations)]
filtered_hrrr = hrrr_df[hrrr_df["station"].isin(stations)]

In [ ]:
clim_div = [
    "St. Lawrence Valley",
    "Great Lakes",
    "Northern Plateau",
    "Champlain Valley",
    "Hudson Valley",
    "Mohawk Valley",
    "Western Plateau",
    "Eastern Pleateau",
    "Coastal",
    "Central Lakes",
]
# clim_div = sorted(clim_div)
image = "/home/aevans/nwp_bias/src/landtype/data/NCEI_logo.png"

In [ ]:
def plot_points(nysm, nwp, clim_div, logo):
    # Remove duplicate stations in HRRR (nwp), keeping only the first instance
    nwp = nwp.drop_duplicates(subset="station", keep="last")
    nysm = nysm.drop_duplicates(subset="station", keep="first")

    # Create plot
    fig = plt.figure(figsize=(24, 16))
    ax = fig.add_subplot(
        1,
        1,
        1,
        projection=crs.LambertConformal(
            central_longitude=-75.0, standard_parallels=(49, 77)
        ),
    )

    # Load the shapefile for boundaries
    shapefile_path = "/home/aevans/nwp_bias/src/machine_learning/notebooks/data/GIS.OFFICIAL_CLIM_DIVISIONS.shp"
    gdf = gpd.read_file(shapefile_path)

    ny_state_boundaries_path = "/home/aevans/nwp_bias/src/landtype/data/State.shx"
    ny_state_boundaries_geo = gpd.read_file(ny_state_boundaries_path).to_crs(epsg=4326)

    ny_bbox = ny_state_boundaries_geo.total_bounds
    gdf_filtered = gdf.cx[ny_bbox[0] : ny_bbox[2], ny_bbox[1] : ny_bbox[3]]
    gdf_filtered = pd.concat([gdf_filtered.iloc[20:29], gdf_filtered.iloc[[32]]])

    # Create a categorical column for plotting
    gdf_filtered["category"] = np.arange(len(gdf_filtered))

    # Plot shapefile with climate divisions (remove the automatic legend)
    gdf_filtered.plot(
        ax=ax,
        transform=crs.PlateCarree(),
        column="category",
        cmap="tab10",
        alpha=0.3,
        legend=False,
    )

    # Create legend for climate divisions using the colors from the 'tab10' colormap and labels from 'clim_div'
    division_patches = [
        mpatches.Patch(
            color=plt.cm.tab10(i / len(gdf_filtered)), alpha=0.3, label=clim_div[i]
        )
        for i in range(len(gdf_filtered))
    ]

    # Add the climate divisions legend
    legend1 = ax.legend(
        handles=division_patches,
        loc="upper right",
        title="Climate Divisions",
        fontsize=12,
    )
    ax.add_artist(legend1)  # Ensure the first legend is added to the plot

    # Set extent for the plot
    ax.set_extent(
        [
            nysm["lon"].min() - 0.1,
            nysm["lon"].max() + 0.1,
            nysm["lat"].min() - 0.1,
            nysm["lat"].max() + 0.1,
        ],
        crs=crs.PlateCarree(),
    )

    # Add features
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linestyle=":", zorder=1)
    ax.add_feature(cfeature.STATES.with_scale("50m"), linestyle=":", zorder=1)
    ax.add_feature(cfeature.LAKES.with_scale("50m"), zorder=1)
    ax.gridlines(
        crs=crs.PlateCarree(),
        draw_labels=True,
        linewidth=2,
        color="black",
        alpha=0.5,
        linestyle="--",
    )

    # Annotate scatter points with station IDs
    for i, row in nysm.iterrows():
        ax.annotate(
            row["station"],
            (row["lon"], row["lat"]),
            textcoords="offset points",
            xytext=(0, 20),
            ha="center",
            fontsize=18,
            color="black",
            transform=crs.PlateCarree(),
        )

    # Plot scatter points
    ax.scatter(
        nysm["lon"],
        nysm["lat"],
        c="blue",
        s=250,
        edgecolors="black",
        transform=crs.PlateCarree(),
        zorder=10,
        label="NYSM",
    )
    ax.scatter(
        nwp["longitude"],
        nwp["latitude"],
        c="orange",
        s=250,
        edgecolors="black",
        transform=crs.PlateCarree(),
        zorder=10,
        label="HRRR",
    )

    # Add plot title
    plt.title(
        f"NCEI New York State Climate Divisions",
        fontsize=24,
    )
    # # Load and add the logo to the lower left
    # logo_img = mpimg.imread(logo)
    # ax.figure.figimage(
    #     logo_img, 50, 50, zorder=20, alpha=0.5
    # )  # Adjust (x, y) position as needed
    # Add legend for station types (NYSM and HRRR)
    legend2 = ax.legend(
        loc="lower right",
        title="Station Type",
        fontsize=12,
    )
    ax.add_artist(legend2)  # Optional: use only if you want to stack both legends

    # Show plot
    plt.show()

In [ ]:
plot_points(filtered_nysm, filtered_hrrr, clim_div, image)